# DS403 · Session 10 — From user understanding to product

> **Work in your own copy** (from day 1): *File → Save a copy in Drive*, then *File → Move* it into **DS403 students → your folder** (your first and last name). Work only there: your teachers follow your progress from that folder. In the exercise cells, **stop a little and think** before you type, and solve them **without consulting generative systems**.


**Fri 30 Oct 2026**, 09:00–12:00. Homework 2 is set today: who talks about AI coding agents on Hacker News, due at session 13.

Today you take the analysis you finished in session 5 and turn it into four things a product can be built from:
a **problem statement**, a **target segment with a success metric**, a **spec** (what and why), and the first **docs and logs** of your project. Then you give your agent its **rules** (the pack in `examples/session-10/rules/`) and run a first **wave of simulated users** on your own page under a **directive** that does not move (section 8). Then you give your agent its **rules** (the pack in `examples/session-10/rules/`) and run a first **wave of simulated users** on your own page under a **directive** that does not move (section 8). Then you give your agent its **rules** (the pack in `examples/session-10/rules/`) and run a first **wave of simulated users** on your own page under a **directive** that does not move (section 8). Then you give your agent its **rules** (the pack in `examples/session-10/rules/`) and run a first **wave of simulated users** on your own page under a **directive** that does not move (section 8). Then you give your agent its **rules** (the pack in `examples/session-10/rules/`) and run a first **wave of simulated users** on your own page under a **directive** that does not move (section 8). Then you give your agent its **rules** (the pack in `examples/session-10/rules/`) and run a first **wave of simulated users** on your own page under a **directive** that does not move (section 8).

The worked example is Ritmo, the course's habit app. Your own project follows the same steps with your own segment.

**How to use this notebook.** Run the setup cell first. Each section below belongs to one block of the session plan
(the block number is in the heading). Cells marked **Your turn** are for you to change. Restart & Run All before you commit.

**AI use today.** Sections 1 and 2 are **AI-free** on purpose: the segment, the metric and the problem statement must be yours.
From section 3 on, use your assistant as you like and say so in your commit message.

In [ ]:
#@title Course setup — run this first (on Colab it prepares Drive, pandas and the data; on a laptop it does nothing)
import os, shutil, subprocess, sys
from pathlib import Path

COURSE_FOLDER = "261019 HSIT Generative Python product development"   # the shared course folder on Google Drive
COURSE_FOLDER_ID = "1z3bZ5zoE6mE7-oyLk1gmBDXGrEVFTlXd"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    from importlib.metadata import version
    if int(version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=3.0,<3.1", "pyarrow"], check=True)
        if "pandas" in sys.modules:   # Colab loaded the old pandas already: only a restart replaces it
            raise SystemExit("pandas 3 is installed. Now: Runtime → Restart session, then run this cell again.")
    import pandas as pd
    home = Path("/content/drive/MyDrive/ds403")
    data = home / "data"
    if not (data / "events.csv").exists():
        drive_root = Path("/content/drive")
        # By the folder's id, never by searching a whole Drive (slow, and it depends on where you keep things):
        # the shortcut in My Drive (wherever you put it, Drive lists its target by id), or the folder itself in My Drive.
        src = next((c for c in (drive_root / ".shortcut-targets-by-id" / COURSE_FOLDER_ID / COURSE_FOLDER / "data",
                                drive_root / "MyDrive" / COURSE_FOLDER / "data") if (c / "events.csv").exists()), None)
        data.mkdir(parents=True, exist_ok=True)
        if src:
            for f in ("events.csv", "audience.csv"):
                shutil.copy(src / f, data / f)
        else:
            # No shortcut: read the two files straight from the course folder (Google asks once for permission).
            from google.colab import auth
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            auth.authenticate_user()
            api = build("drive", "v3")
            def children(parent, extra=""):
                return api.files().list(q=f"'{parent}' in parents and trashed = false{extra}", fields="files(id,name)",
                                        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]
            files = []
            try:   # the folder named 'data' that holds the files (Drive allows two folders with one name)
                for d in children(COURSE_FOLDER_ID, " and name = 'data'"):
                    files = children(d["id"])
                    if any(f["name"] == "events.csv" for f in files):
                        break
                    files = []
            except Exception:
                files = []
            if not files:
                raise FileNotFoundError(f"You cannot open the course folder '{COURSE_FOLDER}' yet: ask your teacher to "
                                        "share it with you, then run this cell again.")
            for f in files:
                if f["name"] in ("events.csv", "audience.csv"):
                    with open(data / f["name"], "wb") as out:
                        down = MediaIoBaseDownload(out, api.files().get_media(fileId=f["id"]))
                        done = False
                        while not done:
                            _, done = down.next_chunk()
        print("The course data is copied to", data)
    # Work from a local folder beside a link to your data on Drive: '../data' then means your ds403/data, exactly as
    # on a laptop (inside the Drive mount itself, '..' does not resolve), and what a notebook saves still lands on Drive.
    local = Path("/content/ds403")
    (local / "work").mkdir(parents=True, exist_ok=True)
    if not (local / "data").exists():
        (local / "data").symlink_to(data, target_is_directory=True)
    os.chdir(local / "work")
    print("Ready on Colab · pandas", pd.__version__, "· data in", data)


## 0 · Setup — load Ritmo the way session 5 left it

In [ ]:
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
if not (DATA / "events.csv").exists():          # the CSVs are generated, never committed
    subprocess.run([sys.executable, str(DATA / "make_data.py")], check=True)


def load_ritmo(data: Path = DATA) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Sessions 2-5 in one function: parse both timestamp formats, drop exact duplicates, drop the bots."""
    audience = pd.read_csv(data / "audience.csv", dtype={"country": "string", "age_band": "string"})
    audience["signup_ts"] = pd.to_datetime(audience["signup_ts"], utc=True)
    events = pd.read_csv(data / "events.csv")
    iso = pd.to_datetime(events["ts"], format="ISO8601", errors="coerce", utc=True)
    old = pd.to_datetime(events["ts"], format="%d/%m/%Y %H:%M:%S", errors="coerce", utc=True)
    events["ts"] = iso.fillna(old)
    events = events.drop_duplicates()
    per_user = events.groupby("user_id").size()
    bots = per_user.index[per_user > 2000]                       # session 5: thousands of events = a bot
    events = events[~events["user_id"].isin(bots)].reset_index(drop=True)
    audience = audience[~audience["user_id"].isin(bots)].reset_index(drop=True)
    return audience, events


audience, events = load_ritmo()
print(f"{len(audience):,} users, {len(events):,} events, {events['ts'].min():%Y-%m-%d} to {events['ts'].max():%Y-%m-%d}")

In [ ]:
def user_table(audience: pd.DataFrame, events: pd.DataFrame, mature_days: int = 35) -> pd.DataFrame:
    """One row per user who signed up at least `mature_days` before the data ends.

    completed_w1: completed at least one habit within 7 calendar days of signup (day 0..6)
    active_w4:    any event on days 28..34 after signup
    premium:      plan == "premium"
    """
    end = events["ts"].max()
    d = events.merge(audience[["user_id", "signup_ts"]], on="user_id", validate="many_to_one")
    d["day"] = (d["ts"].dt.normalize() - d["signup_ts"].dt.normalize()).dt.days
    first_complete = d.loc[d["event"].eq("complete_habit")].groupby("user_id")["day"].min()
    active_w4 = d.loc[d["day"].between(28, 34), "user_id"].unique()
    users = audience.assign(
        completed_w1=audience["user_id"].map(first_complete).lt(7),
        active_w4=audience["user_id"].isin(active_w4),
        premium=audience["plan"].eq("premium"),
    )
    return users[users["signup_ts"] <= end - pd.Timedelta(days=mature_days)].reset_index(drop=True)


users = user_table(audience, events)
print(f"{len(users):,} users signed up at least 35 days before the data ends (the 'mature' users)")
users.head()

## 1 · Block 2 — From the analysis to a segment and a metric (live code, follow along)

Session 5 told us **who** the users are and **where they struggle**. A product needs one more step:
*which* group we build for (the **target segment**) and *which number* tells us we helped them (the **success metric**).

We look at the first week, because that is where users are won or lost.

In [ ]:
by_channel = (
    users.groupby("acquisition_channel")
    .agg(users=("user_id", "size"),
         completed_w1=("completed_w1", "mean"),
         active_w4=("active_w4", "mean"),
         premium=("premium", "mean"))
    .sort_values("completed_w1")
)
by_channel.round(3)

Two channels sit at the bottom on every column: **paid_social** and **influencer**. Together they are a large share of
signups. That is a candidate segment. Now the link between the first week and week 4:

In [ ]:
link = users.groupby("completed_w1")[["active_w4", "premium"]].mean()
link.index = link.index.map({False: "no habit completed in week 1", True: "completed a habit in week 1"})
link.round(3)

**Careful.** This is an *association*. Users who complete a habit in week 1 may simply be the motivated ones.
The data cannot tell us that helping someone complete a habit *causes* them to stay. That is exactly the **bet**
the MVP will test (block 3).

In [ ]:
segment = users["acquisition_channel"].isin(["paid_social", "influencer"])
summary = pd.DataFrame({
    "users": [segment.sum(), (~segment).sum()],
    "share_of_signups": [segment.mean(), 1 - segment.mean()],
    "completed_w1": [users.loc[segment, "completed_w1"].mean(), users.loc[~segment, "completed_w1"].mean()],
    "active_w4": [users.loc[segment, "active_w4"].mean(), users.loc[~segment, "active_w4"].mean()],
}, index=["paid_social + influencer", "everyone else"])
summary.round(3)

In [ ]:
# Does the gap hold inside every device? (Session 5: cut every metric by segment, watch for Simpson.)
pd.crosstab(users["device"], segment.map({True: "target", False: "rest"}),
            values=users["completed_w1"], aggfunc="mean").round(3)

The gap holds on every device, so it is not a device-mix effect.

**The worked example's three sentences** (compare with the numbers above):

* **Target segment:** new users who arrive from paid social or influencer campaigns.
* **Success metric (leading):** share of the segment who complete at least one habit within 7 days of signup.
* **Guardrail metric (lagging):** share of the segment active in week 4 (days 28–34).

**Your notes · Block 2** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 2 · Block 2b — Your segment and your metric (hands-on, AI-free)

Use your session 5 analysis. If you are building for Ritmo, choose a different segment from the worked example
(for example: web users, or users who never open the app after signup). If your project uses other data, write the
same function for your data.

1. Define your segment as a boolean mask over `users`.
2. Define your metric as a function of a group of users that returns one number between 0 and 1.
3. Compute it for the segment and for everyone else.

In [ ]:
# Your turn: change the mask and the metric. The defaults are an example (web users, first-week completion).
my_segment = users["device"].eq("web")

def my_metric(group: pd.DataFrame) -> float:
    """Share of the group who completed a habit within 7 days of signup."""
    return float(group["completed_w1"].mean())

print(f"segment size: {my_segment.sum():,} users ({my_segment.mean():.1%} of mature signups)")
print(f"metric in segment:   {my_metric(users[my_segment]):.1%}")
print(f"metric outside it:   {my_metric(users[~my_segment]):.1%}")

In [ ]:
# A metric you cannot move is not a product metric. Check it is not already at the ceiling or the floor,
# and that the segment is big enough to measure a change of 5 points.
def check_metric(segment_mask: pd.Series, metric) -> list[str]:
    problems = []
    value = metric(users[segment_mask])
    if value > 0.9 or value < 0.02:
        problems.append(f"metric is {value:.1%}: too close to the ceiling or the floor to move")
    n = int(segment_mask.sum())
    # rough 95% margin of error for a proportion: 1.96 * sqrt(p(1-p)/n)
    moe = 1.96 * (value * (1 - value) / max(n, 1)) ** 0.5
    if moe > 0.05:
        problems.append(f"segment of {n} users: margin of error {moe:.1%} is larger than a 5-point change")
    return problems or ["ok: the metric can move and a 5-point change is measurable"]

check_metric(my_segment, my_metric)

**Your notes · Block 2b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 3 · Block 3 — From the problem statement to a spec (hands-on)

A spec-kit specification says **what** and **why**, never **how**. No library names, no tables, no endpoints:
those belong to the plan. Write your spec draft in the cell below (it is just a string), then lint it.

In [ ]:
spec = """
# Feature: First habit in the first week

## Why
New users from paid social and influencer campaigns are 45% of signups. Only 28% of them complete a habit in
their first 7 days, against 46% of everyone else, and users who complete one in week 1 are far more often still
active in week 4. We bet that helping these users complete one habit in week 1 raises their week-4 activity.

## Who
New users whose acquisition channel is paid social or influencer, in their first 7 days.

## What
- A new user in the segment who has not completed a habit by day 2 receives one suggestion of a small habit.
- The suggestion can be accepted in one step.

## Success
- Primary: share of the segment completing at least one habit within 7 days rises from 28% to 34%.
- Guardrail: week-4 activity of the segment does not fall.

## Out of scope
- Paid plans and pricing.
- Users from other channels.
"""

HOW_WORDS = ["pandas", "fastapi", "streamlit", "gradio", "database", "sql", "endpoint", "api", "json",
             "python", "class ", "function", "table", "csv", "docker", "react"]

def lint_spec(text: str) -> list[str]:
    """Flag what a spec should not contain (the how) and what it must contain (the why, a measurable success)."""
    low = text.lower()
    findings = [f"'how' word in a what/why spec: {w.strip()!r}" for w in HOW_WORDS if w in low]
    for heading in ["## why", "## who", "## what", "## success", "## out of scope"]:
        if heading not in low:
            findings.append(f"missing section: {heading[3:]}")
    success = low.split("## success", 1)[-1]
    if not any(ch.isdigit() for ch in success):
        findings.append("success section has no number: is it measurable?")
    return findings or ["clean: what and why, with a measurable success"]

lint_spec(spec)

In [ ]:
# The same linter on a spec that slipped into the how:
lint_spec("""
## Why
Users churn.
## What
A FastAPI endpoint reads the events table with pandas and returns JSON.
## Success
Users are happier.
""")

**Your turn.** Replace the text of `spec` with your own project's spec and run the linter until it is clean.
Then paste it into `/speckit.specify` in your repository (session 6), and commit it before any code.

**Your notes · Block 3** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 4 · Block 14 — README first, and your first ADR (hands-on)

Write the README **as if the product already existed**: what it is, how to run it, what it does not do.
Then record one decision as an ADR: context, decision, consequences. One decision per file.

In [ ]:
readme = """
# ritmo-first-week

Finds new Ritmo users from paid social and influencer campaigns who have not completed a habit by day 2,
and suggests one small habit to each of them.

## Run it
    python -m venv .venv && source .venv/bin/activate
    pip install -r requirements.txt
    python data/make_data.py
    fastapi dev app/main.py        # the API, http://127.0.0.1:8000/docs
    streamlit run ui/app.py        # the page for the product team

## What it does not do
- It does not send messages: it lists users and suggestions; sending is someone else's system.
- It does not cover users from other channels.
- It does not prove the suggestion works: the A/B test does (see docs/adr/0002).

## Docs
- spec: specs/001-first-habit/spec.md
- decisions: docs/adr/
- agent instructions: AGENTS.md
"""

adr = """
# ADR-0001: Compute the first-week metric in a daily batch, not in real time

Status: Accepted
Date: 2026-10-30

## Context
The success metric is the share of the segment completing a habit within 7 days of signup. It changes once
per day at most. events.csv arrives as a daily export. Nobody acts on it faster than daily.

## Decision
We compute the metric once a day from the export, in one function with tests, and serve the stored result.

## Consequences
- Simple: one function, no stream processing, easy to test against the course dataset.
- The number can be up to 24 hours old; the product page says so.
- If we later need same-day suggestions, this ADR is superseded by a new one, not edited.
"""

def check_readme(text: str) -> list[str]:
    low = text.lower()
    need = {"what it is (first paragraph)": len(low.strip().splitlines()) > 2,
            "how to run it": "## run" in low or "## usage" in low or "## quick" in low,
            "what it does not do": "not" in low and ("does not" in low or "out of scope" in low)}
    return [f"missing: {k}" for k, ok in need.items() if not ok] or ["README ok"]

def check_adr(text: str) -> list[str]:
    low = text.lower()
    missing = [h for h in ["## context", "## decision", "## consequences"] if h not in low]
    out = [f"missing section: {h[3:]}" for h in missing]
    if "status:" not in low:
        out.append("no status (proposed / accepted / superseded)")
    if low.count("## decision") > 1:
        out.append("more than one decision: split into two ADRs")
    return out or ["ADR ok"]

check_readme(readme), check_adr(adr)

**Your turn.** Write your own `readme` and `adr` strings for your project (one real decision you have already made,
for example: *which data*, *batch or live*, *Streamlit or Gradio*, *SQLite or files*). When both checks pass,
copy them into `README.md` and `docs/adr/0001-<slug>.md` in your repository, in the same PR as the code they describe.

**Your notes · Block 14** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 5 · Block 15 — Logging from day one (live code)

`print` is for you, today. A **structured log** is for you next week, for your teammate, and for the agent that
reads your logs: one JSON object per line, with named fields. Python's `logging` does this with a small formatter.

In [ ]:
import io
import json
import logging
import time


class JsonFormatter(logging.Formatter):
    """One JSON object per line: time, level, logger, message, and any fields passed in `extra={...}`."""
    STANDARD = set(vars(logging.makeLogRecord({})))

    def format(self, record: logging.LogRecord) -> str:
        entry = {"ts": self.formatTime(record, "%Y-%m-%dT%H:%M:%S"), "level": record.levelname,
                 "logger": record.name, "msg": record.getMessage()}
        entry.update({k: v for k, v in vars(record).items() if k not in self.STANDARD})
        return json.dumps(entry, default=str)


log_buffer = io.StringIO()                     # in a real project: sys.stdout or a file
handler = logging.StreamHandler(log_buffer)
handler.setFormatter(JsonFormatter())
log = logging.getLogger("ritmo.metrics")
log.handlers[:] = [handler]
log.setLevel(logging.INFO)
log.propagate = False


def first_week_completion(users: pd.DataFrame, channels: list[str]) -> float:
    started = time.perf_counter()
    group = users[users["acquisition_channel"].isin(channels)]
    if group.empty:
        log.warning("empty_segment", extra={"channels": channels})
        return float("nan")
    value = float(group["completed_w1"].mean())
    log.info("metric_computed", extra={"metric": "first_week_completion", "channels": channels,
                                       "n_users": len(group), "value": round(value, 4),
                                       "ms": round(1000 * (time.perf_counter() - started), 2)})
    return value


first_week_completion(users, ["paid_social", "influencer"])
first_week_completion(users, ["organic"])
first_week_completion(users, ["tiktok"])          # a channel that does not exist
print(log_buffer.getvalue())

**Your notes · Block 15** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 6 · Block 15b — Read your own logs back (hands-on)

Because each line is JSON, your logs are **data**: the same pandas you used all course reads them.

In [ ]:
logs = pd.read_json(io.StringIO(log_buffer.getvalue()), lines=True)
logs[["level", "msg", "channels", "n_users", "value", "ms"]]

In [ ]:
# Your turn:
# 1. Add a log line inside check_metric (section 2) with the segment size and the margin of error.
# 2. Call first_week_completion for every channel in a loop, then answer from the logs only:
#    which call was slowest, and which segment was smallest?
for ch in sorted(users["acquisition_channel"].unique()):
    first_week_completion(users, [ch])
logs = pd.read_json(io.StringIO(log_buffer.getvalue()), lines=True)
done = logs[logs["msg"].eq("metric_computed")]
print("smallest segment:", done.loc[done["n_users"].idxmin(), "channels"], int(done["n_users"].min()), "users")
print("warnings:", int(logs["level"].eq("WARNING").sum()))

**What not to log.** `user_id` linked to behaviour is personal data (session 5, GDPR). Log counts, segments and
timings, not who did what. If you must log an id for debugging, log it at DEBUG level and never ship DEBUG logs.

**Your notes · Block 15b** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 7 · Block 16 — Generate, then own: a Streamlit page (hands-on, assistant allowed)

Ask your assistant for a Streamlit page that shows `first_week_completion` for a channel chosen in a select box.
It will look roughly like the code below. **Do not run it in this notebook** (Streamlit runs as a script:
`streamlit run app.py`). Read it, then answer the questions in the next cell *without the assistant*.

```python
# app.py
import pandas as pd
import streamlit as st

from ritmo.metrics import load_ritmo, user_table   # your own module

@st.cache_data                                      # without this, every click reloads 18 MB of CSV
def load_users() -> pd.DataFrame:
    audience, events = load_ritmo()
    return user_table(audience, events)

st.title("Ritmo · first-week habit completion")
users = load_users()
channel = st.selectbox("Acquisition channel", sorted(users["acquisition_channel"].unique()))
group = users[users["acquisition_channel"] == channel]
st.metric("Completed a habit in week 1", f"{group['completed_w1'].mean():.1%}", help=f"{len(group)} users")
st.bar_chart(users.groupby("acquisition_channel")["completed_w1"].mean())
```

In [ ]:
# Streamlit's model in ten lines: every interaction reruns the whole script from the top.
# Here a "rerun" is a call of page(); the cache decides whether the expensive load runs again.
loads = {"count": 0}
_cache = {}

def load_users_cached():
    if "users" not in _cache:                 # what @st.cache_data does, minus hashing the arguments
        loads["count"] += 1
        _cache["users"] = users                # pretend this is the 18 MB read
    return _cache["users"]

def page(channel: str) -> str:
    u = load_users_cached()                    # line 1 of the script runs on EVERY rerun
    g = u[u["acquisition_channel"] == channel]
    return f"{channel}: {g['completed_w1'].mean():.1%} of {len(g)} users"

for choice in ["organic", "paid_social", "influencer", "organic"]:   # four clicks on the select box
    print(page(choice))
print("expensive loads:", loads["count"], "for 4 reruns")

**Own it — answer in your own words (AI-free, 5 minutes), then compare with a neighbour:**

1. You change the select box. Which lines of `app.py` run again? *(All of them, from the top.)*
2. What would happen to the page's speed without `@st.cache_data`, and why?
3. `load_users()` returns a DataFrame. If a line below it modified `users` in place, would the next rerun see the change? *(No: `st.cache_data` returns a copy each time.)*
4. Which line would you change to show week-4 activity instead, and which test would tell you it is right?

If you cannot answer one of these for a module in your project, you do not own that module yet.

**Your notes · Block 16** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## 8 · Block 17 — A wave of simulated users on your page (hands-on)

Part 3 of the deck, in code. The **directive** stays fixed. **Personas** from your segments look at a screen and
return **findings** as JSON: what they tried, what happened, the problem, a severity from 1 to 4, and which of
Nielsen's ten heuristics it breaks. A **triage** keeps at most five changes that serve the directive, and rejects
the ones that contradict it.

* **Outside** the notebook: paste `ux_critic.md` and a screenshot into Claude or ChatGPT, once per persona; paste the
  JSON back into `outside_answer` below; `check_findings` refuses an answer that is not a wave you can count.
* **Inside**: `python simulate_wave.py …` in your terminal (the deck's lab slide), or the last cell with your key.
* **Your own directive**: copy `directive_template.md` to `directive.md` in your repository and write it yourself;
  the cells below use Ritmo's.

First, read a real wave: the one we ran on our own product on 29 October (`examples/session-10/demo_wave/`).

In [ ]:
import os
import re

_drive = Path("/content/drive")                  # on Colab: the course folder, by its id (shortcut) or in My Drive
_cf, _cid = globals().get("COURSE_FOLDER", ""), globals().get("COURSE_FOLDER_ID", "")
EX = next((p for p in (Path("../examples/session-10"), Path("examples/session-10"),
                       _drive / ".shortcut-targets-by-id" / _cid / _cf / "examples" / "session-10",
                       _drive / "MyDrive" / _cf / "examples" / "session-10") if _cf is not None and p.exists()), None)
print("examples:", EX.resolve() if EX else "not found: the cells below use the copy inside this notebook")

NIELSEN = ["visibility of system status", "match between system and the real world", "user control and freedom",
           "consistency and standards", "error prevention", "recognition rather than recall",
           "flexibility and efficiency of use", "aesthetic and minimalist design",
           "help users recognize, diagnose, and recover from errors", "help and documentation"]

# Ritmo's directive, from section 3's spec: Why and Success, plus what must never change.
DIRECTIVE = """Ritmo helps a new user keep one small daily habit, starting in their first week.
Must not change without the product owner: (1) one habit at a time, never a list to choose from on day 1;
(2) no shaming, no streak loss shown in red; (3) every suggestion can be accepted in one step;
(4) we measure first-week completion of the paid-social and influencer segment, and week-4 activity as a guardrail."""

PERSONAS = [
    {"name": "Nadia", "segment": "paid social, first week", "device": "phone, between classes",
     "goal": "try the app the ad promised in under a minute", "mistakes": ["skips text", "taps the first button"]},
    {"name": "Tomás", "segment": "influencer follower", "device": "phone, evenings",
     "goal": "copy the routine the influencer showed", "mistakes": ["expects a list of habits", "leaves half-way"]},
    {"name": "Ingrid", "segment": "organic, month two", "device": "laptop",
     "goal": "see whether the app is worth paying for", "mistakes": ["reads every number literally"]},
]

In [ ]:
def check_findings(raw, persona: str = "?", run: int = 1) -> pd.DataFrame:
    """A wave you can count: a JSON list of findings, each with the fields below, a known heuristic and severity 1-4."""
    items = json.loads(raw) if isinstance(raw, str) else raw
    if isinstance(items, dict):
        items = items.get("findings", [])
    need = ["screen", "what_i_tried", "what_happened", "problem", "severity", "heuristic", "suggestion"]
    rows, refused = [], []
    for i, f in enumerate(items):
        missing = [k for k in need if not str(f.get(k, "")).strip()]
        h = str(f.get("heuristic", "")).strip().lower()
        sev = f.get("severity")
        if missing:
            refused.append((i, f"missing {missing}"))
        elif not any(h.startswith(n[:18]) for n in NIELSEN):
            refused.append((i, f"heuristic not one of Nielsen's ten: {h!r}"))
        elif sev not in (1, 2, 3, 4):
            refused.append((i, f"severity must be 1-4, got {sev!r}"))
        else:
            rows.append({"persona": f.get("persona", persona), "run": f.get("run", run), **{k: f[k] for k in need}})
    for i, why in refused:
        print(f"refused finding {i}: {why}")
    return pd.DataFrame(rows)


# Two findings, one good and one the check refuses:
check_findings("""[
 {"screen": "today", "what_i_tried": "accept the suggested habit", "what_happened": "a list of 12 habits opened",
  "problem": "I had to choose before I understood", "severity": 3, "heuristic": "Recognition rather than recall",
  "suggestion": "show one habit with one Accept button"},
 {"screen": "today", "problem": "ugly", "severity": 9}
]""", persona="Nadia")

**Read a real wave.** Our own product's screens, four fictional personas, each run twice, then a triage against our
directive (`demo_wave/README.md` tells the whole story). Which heuristics come up most? Do the two runs of the same
persona find the same problems? A finding that only one run found is a reason to look, not a fact.

In [ ]:
DEMO = None
if EX is not None and (EX / "demo_wave").exists():
    parts = []
    for f in sorted((EX / "demo_wave").glob("stage*/findings.json")):
        d = check_findings(f.read_text())
        parts.append(d.assign(stage=f.parent.name))
    DEMO = pd.concat(parts, ignore_index=True) if parts else None
if DEMO is None:
    print("demo_wave not found: open examples/session-10/demo_wave/ in the course folder")
else:
    print(f"{len(DEMO)} findings, {DEMO['persona'].nunique()} personas, stages: {sorted(DEMO['stage'].unique())}")
    display(DEMO.pivot_table(index="heuristic", columns="stage", values="severity", aggfunc="count", fill_value=0))

In [ ]:
def screen_key(screen: str) -> str:
    """The first id a finding names: a file (stage3_day_one.jpg) or a situation (US.C.87.04), not its free text."""
    m = re.match(r"[\w.]+", str(screen).strip())
    key = m.group(0) if m else str(screen)
    return ".".join(key.split(".")[:4]) if key.startswith("US.") else key


def run_agreement(d: pd.DataFrame) -> pd.DataFrame:
    """For each persona and stage: how many (screen, heuristic) pairs both runs found, of all pairs either found."""
    out = []
    for (stage, persona), g in d.groupby(["stage", "persona"]):
        runs = [set(zip(r["screen"].map(screen_key), r["heuristic"])) for _, r in g.groupby("run")]
        if len(runs) >= 2:
            both, either = set.intersection(*runs), set.union(*runs)
            out.append({"stage": stage, "persona": persona, "both runs": len(both), "either run": len(either),
                        "agreement": round(len(both) / len(either), 2) if either else None})
    return pd.DataFrame(out)


if DEMO is not None:
    display(run_agreement(DEMO))

**Your turn — outside the notebook.** Take a screenshot of your Streamlit page from section 7 (or your mockup).
Print the prompt below, paste it with the screenshot into Claude or ChatGPT, once per persona (change `PERSONAS` to
your own segments first). Paste the JSON it returns into `outside_answer` and run the check. Then paste every
finding, your directive and `wave_triage.md` into a new chat: at most five changes, each accepted, deferred or
rejected with the line of the directive it serves or breaks.

In [ ]:
_FALLBACK = """---8<---
You are a real person trying to use a product: {{PRODUCT}}
Who you are: {{PERSONA}}
Walk what you are shown as yourself, mistakes included. Reply with JSON only:
{"findings": [{"screen": "...", "what_i_tried": "...", "what_happened": "...", "problem": "...",
               "severity": 1-4, "heuristic": "one of Nielsen's ten, by name", "suggestion": "..."}]}"""
_critic = (EX / "ux_critic.md").read_text() if EX is not None and (EX / "ux_critic.md").exists() else _FALLBACK
CRITIC = ("\n" + _critic).split("\n---8<---\n", 1)[-1].strip()   # the prompt itself is below the cut line


def critic_prompt(persona: dict, screen: str) -> str:
    """ux_critic.md with the product (here: the directive) and one persona filled in."""
    return (CRITIC.replace("{{PRODUCT}}", DIRECTIVE).replace("{{PERSONA}}", json.dumps(persona, ensure_ascii=False))
            + f"\n\nWhat you are shown: {screen}. Walk it as {persona['name']} and report.")


print(critic_prompt(PERSONAS[0], "the attached screenshot of my Streamlit page"))

outside_answer = """[]"""          # paste the JSON here
mine = check_findings(outside_answer, persona=PERSONAS[0]["name"])
mine

In [ ]:
# Inside: the same prompt, sent with your screenshot. Needs ANTHROPIC_API_KEY in Colab's secrets (the key icon)
# and a screenshot uploaded as my_page.png. Read the code before you run it; it costs a few cents.
import base64

key = os.environ.get("ANTHROPIC_API_KEY")
try:
    from google.colab import userdata
    key = key or userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
shot = Path("my_page.png")
if key and shot.exists():
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    found = []
    for p in PERSONAS:
        for run in (1, 2):                 # twice: one run is not the answer
            msg = client.messages.create(model="claude-sonnet-5-5", max_tokens=2000, messages=[{"role": "user", "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/png",
                                             "data": base64.b64encode(shot.read_bytes()).decode()}},
                {"type": "text", "text": critic_prompt(p, "the screenshot above")}]}])
            text = msg.content[0].text
            found.append(check_findings(text[text.find("{"): text.rfind("}") + 1], persona=p["name"], run=run))
    wave = pd.concat(found, ignore_index=True)
    display(wave.sort_values("severity", ascending=False).head(15))
else:
    print("skipped: no key or no my_page.png; use the outside route above")

**Your notes · Block 17** — what you think about this task: what was clear, what was hard or confusing, what you would do differently, and your feedback for the teachers. Your own words, a few lines are enough; your teachers read them in your folder.

*Write below this line (double-click this cell, then Shift+Enter):*

…


## If there is time

Extra work for when the session runs ahead of plan; each takes about 10 minutes and stands alone. Your teacher picks one, or try them at home.

* **Can your experiment see the change?** (after block 2): A success metric taken from your own analysis also needs a target that an experiment of realistic size can [Session 10 notebook](https://colab.research.google.com/drive/1P9poI37m2_lrAF2hgc1BJTIG3Jmz27Sj)
* **The kill criterion, written first** (after block 3): An MVP tests a bet, so it needs a kill criterion written before the build: the result that would prove the
* **ADR or not?** (after block 14): Write an ADR for each decision that is expensive to reverse: context, decision, consequences, one decision
* **One request, one id** (after block 15): Structured logs carry named fields from the first commit. A request id groups every line of one request, and
* **The rule that was never cited** (after block 7): A rule earns its place by being cited and followed; prune the ones that are not, and give a broken one a
